<a href="https://colab.research.google.com/github/alparm3386/llm-finetune-lora/blob/master/notebooks/eval_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Before/after evaluation — Gemma 4 E2B (Hungarian text→JSON extraction)

Thin Colab wrapper around `src/evaluate.py` (dev-plan step 3.5). All the actual
logic (model loading, generation, metrics, reporting) lives in the script, not
here — this notebook installs Unsloth + outlines, fetches the repo + the
trained adapter (step 3.6 output) + the real eval set, and invokes the CLI.

**Runtime:** Runtime → Change runtime type → T4 GPU, before running anything below.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install unsloth outlines

## Repo

Clone the repo (set `REPO_URL` below) so `src/` is available.

In [ ]:
REPO_URL = "https://github.com/alparm3386/hu-llm-finetune-lora.git"  # noqa: set this
REPO_DIR = "llm-finetune-lora"

!git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

## Trained adapter

The LoRA adapter from step 3.6 (`train_colab.ipynb`'s `outputs/adapter.zip`) is
not part of this clone. Upload it here, or skip this cell and instead set
`ADAPTER_DIR` below to a Hugging Face Hub adapter repo id (step 3.7 output) —
`evaluate.py --adapter` accepts either a local path or a Hub id.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # select outputs/adapter.zip from the training run
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall("outputs/adapter")

## Real eval set

`data/eval/{medical,business,technology}.jsonl` is the hand-labeled set (step
3.5.0, `data/eval/README.md`) — tracked in git, so it comes with the clone if
already populated and committed locally. If you're iterating before that's
committed, upload it the same way as the adapter.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # optional: select a zip of data/eval/*.jsonl
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall("data/eval")

## Shakeout

Quick run on a handful of examples to confirm the pipeline works end-to-end
(model loading, both decode modes, reporting) before the full eval set.

In [ ]:
ADAPTER_DIR = "outputs/adapter"  # local path, or a HF Hub adapter repo id (step 3.7)

!python src/evaluate.py --adapter "$ADAPTER_DIR" --limit 3 --out results/shakeout

## Full evaluation

Runs all four cells of the 2x2 ({base, fine-tuned} x {prompt-only,
+structured decoding}) over the full eval set.

In [ ]:
!python src/evaluate.py --adapter "$ADAPTER_DIR" --out results

`results/eval_results.json` (full breakdown) and `results/eval_table.md` (the
2x2 markdown table for the README, step 3.8) are written locally. Download
them before the Colab runtime recycles.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("eval_results", "zip", "results")
files.download("eval_results.zip")